# Titanic Survival Prediction

**Goal:** Build a binary classification model to predict passenger survival using the Kaggle Titanic dataset.

**Pipeline:**
1. Exploratory data analysis
2. Feature engineering and data cleaning
3. Model training — Random Forest with cross-validation
4. Hyperparameter tuning — GridSearchCV
5. Model evaluation — accuracy, F1, recall, ROC-AUC, confusion matrix
6. Feature importance analysis

**Dataset:** [Kaggle Titanic — Machine Learning from Disaster](https://www.kaggle.com/c/titanic)

---

## 0. Configuration

In [ ]:
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"
RANDOM_STATE = 29

## 1. Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import (
    accuracy_score, f1_score, recall_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, roc_auc_score, RocCurveDisplay
)
from sklearn.preprocessing import StandardScaler

pd.set_option("future.no_silent_downcasting", True)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

## 2. Load Data

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)

print(f"Train shape : {train.shape}")
print(f"Test shape  : {test.shape}")
train.head()

**Feature descriptions:**

| Feature | Description |
|---------|-------------|
| `Survived` | Target — 0 = No, 1 = Yes |
| `Pclass` | Ticket class — 1 = 1st, 2 = 2nd, 3 = 3rd |
| `Sex` | Passenger sex |
| `Age` | Age in years |
| `SibSp` | # of siblings / spouses aboard |
| `Parch` | # of parents / children aboard |
| `Fare` | Passenger fare |
| `Cabin` | Cabin number |
| `Embarked` | Port — C = Cherbourg, Q = Queenstown, S = Southampton |

## 3. Exploratory Data Analysis

In [ ]:
# Missing value summary
missing = train.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values (train):")
print(missing.to_string())

In [ ]:
# Survival rate by sex and passenger class
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.pointplot(x="Sex", y="Survived", hue="Pclass",
              data=train, ax=axes[0])
axes[0].set_title("Survival Rate by Sex and Class")
axes[0].set_ylabel("Survival Rate")

sns.histplot(data=train, x="Age", hue="Survived",
             bins=30, kde=True, ax=axes[1])
axes[1].set_title("Age Distribution by Survival")

plt.tight_layout()
plt.show()

In [ ]:
# Overall survival rate
survival_rate = train["Survived"].mean()
print(f"Overall survival rate: {survival_rate:.1%}")
print(f"Survived    : {train['Survived'].sum()}")
print(f"Did not survive: {(train['Survived'] == 0).sum()}")

## 4. Feature Engineering and Data Cleaning

A shared cleaning function is applied to both train and test sets to ensure
identical preprocessing — a common source of train/test leakage bugs.

In [ ]:
# Title mapping — consolidate rare titles to reduce cardinality
TITLE_MAP = {
    "Ms.": "Miss.", "Mlle.": "Miss.",
    "Mme.": "Mrs.",
    "Dr.": "Rare", "Major.": "Rare", "Lady.": "Rare",
    "Sir.": "Rare", "Col.": "Rare", "Capt.": "Rare",
    "Countess.": "Rare", "Jonkheer.": "Rare",
    "Dona.": "Rare", "Don.": "Rare", "Rev.": "Rare"
}

def extract_title(name: str) -> str:
    """Extract honorific title from passenger name string."""
    match = re.findall(r"\w+[.]", name)
    return match[0] if match else "Unknown"


def clean_dataset(df: pd.DataFrame,
                  title_survival_map: dict = None,
                  is_train: bool = True) -> pd.DataFrame:
    """
    Apply all cleaning and feature engineering steps.

    Parameters
    ----------
    df : raw DataFrame (train or test)
    title_survival_map : dict mapping title → mean survival rate.
                         Must be fitted on train and passed to test.
    is_train : True for training set (includes Survived column)
    """
    df = df.copy()

    # 1. Drop Cabin — too sparse (>77% missing)
    df = df.drop(columns=["Cabin"])

    # 2. Impute Embarked with mode (only 2 missing)
    df["Embarked"] = df["Embarked"].fillna("S")

    # 3. Encode Sex as binary (0 = male, 1 = female)
    df["Sex"] = df["Sex"].map({"male": 0, "female": 1})

    # 4. Extract title from Name and consolidate rare titles
    df["Title"] = df["Name"].map(extract_title).replace(TITLE_MAP)

    # 5. Impute Age using median within each title group
    df["Age"] = df.groupby("Title")["Age"].transform(
        lambda x: x.fillna(x.median())
    )

    # 6. One-hot encode Embarked (C = baseline, dropped)
    df = pd.get_dummies(df, columns=["Embarked"], drop_first=True)

    # 7. Encode Title as mean survival rate (target encoding)
    #    Fitted on train only — passed as argument for test set
    if is_train:
        title_survival_map = df.groupby("Title")["Survived"].mean().to_dict()
    df["Title_encoded"] = df["Title"].map(title_survival_map)

    # 8. Select final feature columns
    feature_cols = [
        "Pclass", "Sex", "Age", "SibSp", "Parch",
        "Fare", "Embarked_Q", "Embarked_S", "Title_encoded"
    ]
    if is_train:
        feature_cols = ["Survived"] + feature_cols

    # Ensure Embarked_Q and Embarked_S exist (may be absent in small test sets)
    for col in ["Embarked_Q", "Embarked_S"]:
        if col not in df.columns:
            df[col] = 0

    return df[feature_cols], title_survival_map

In [ ]:
# Clean train first to fit the title survival map
train_clean, title_survival_map = clean_dataset(train, is_train=True)

# Apply same map to test (prevents data leakage)
test_clean, _ = clean_dataset(test,
                               title_survival_map=title_survival_map,
                               is_train=False)

print("Train clean shape:", train_clean.shape)
print("Test clean shape :", test_clean.shape)
print("\nRemaining nulls (train):")
print(train_clean.isna().sum()[train_clean.isna().sum() > 0])
train_clean.head()

## 5. Train / Validation Split

In [ ]:
X = train_clean.drop("Survived", axis=1)
y = train_clean["Survived"]

# Hold out 20% as a validation set for final evaluation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Validation set: {X_val.shape[0]} samples")

## 6. Baseline Model — Random Forest with Cross-Validation

5-fold cross-validation on the training set gives an unbiased estimate of
generalization performance before any hyperparameter tuning.

In [ ]:
rf_baseline = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=RANDOM_STATE
)

cv_scores = cross_val_score(
    rf_baseline, X_train, y_train, cv=5, scoring="accuracy"
)

print(f"CV Accuracy (5-fold): {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"Per-fold scores     : {[round(s, 3) for s in cv_scores]}")

## 7. Hyperparameter Tuning — GridSearchCV

Grid search over key Random Forest hyperparameters using 5-fold CV.
The best configuration is then used for all downstream evaluation.

In [ ]:
param_grid = {
    "n_estimators" : [100, 200, 300],
    "max_depth"    : [4, 6, 8, None],
    "min_samples_split": [2, 5, 10],
    "max_features" : ["sqrt", "log2"]
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=RANDOM_STATE),
    param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"\nBest parameters : {grid_search.best_params_}")
print(f"Best CV accuracy: {grid_search.best_score_:.3f}")

rf_best = grid_search.best_estimator_

## 8. Model Evaluation on Validation Set

The held-out validation set is used exactly once — here — for final
performance reporting.

In [ ]:
y_pred  = rf_best.predict(X_val)
y_probs = rf_best.predict_proba(X_val)[:, 1]

metrics = {
    "Accuracy" : accuracy_score(y_val, y_pred),
    "F1 Score" : f1_score(y_val, y_pred),
    "Precision": precision_score(y_val, y_pred),
    "Recall"   : recall_score(y_val, y_pred),
    "ROC-AUC"  : roc_auc_score(y_val, y_probs)
}

print("── Validation Set Performance ──")
for metric, value in metrics.items():
    print(f"  {metric:<12}: {value:.3f}")

### 8.1 Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_val, y_pred,
    display_labels=["Did Not Survive", "Survived"],
    cmap="Blues",
    ax=ax
)
ax.set_title("Confusion Matrix — Validation Set")
plt.tight_layout()
plt.show()

### 8.2 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
RocCurveDisplay.from_predictions(
    y_val, y_probs,
    name=f"Random Forest (AUC = {metrics['ROC-AUC']:.3f})",
    ax=ax
)
ax.plot([0, 1], [0, 1], "k--", linewidth=0.8, label="Random classifier")
ax.set_title("ROC Curve — Validation Set")
ax.legend()
plt.tight_layout()
plt.show()

## 9. Feature Importance

Mean decrease in impurity (MDI) importance from the tuned Random Forest.
Higher values indicate greater contribution to reducing classification error.

In [ ]:
importance_df = pd.DataFrame({
    "Feature"   : X_train.columns,
    "Importance": rf_best.feature_importances_
}).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(importance_df["Feature"], importance_df["Importance"],
        color="steelblue", edgecolor="white")
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_title("Random Forest — Feature Importance")
plt.tight_layout()
plt.show()

print(importance_df.sort_values("Importance", ascending=False).to_string(index=False))

## 10. Final Model — Refit on Full Training Data

After validation, refit the tuned model on the entire training set
before generating predictions on the held-out test set.

In [ ]:
rf_final = RandomForestClassifier(
    **grid_search.best_params_,
    random_state=RANDOM_STATE
)
rf_final.fit(X, y)

# Generate test set predictions
test_preds = rf_final.predict(test_clean)

# Build submission file
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived"   : test_preds
})

submission.to_csv("submission.csv", index=False)
print(f"Submission saved: {len(submission)} predictions")
print(f"Predicted survivors: {test_preds.sum()} / {len(test_preds)} ({test_preds.mean():.1%})")
submission.head(10)

## 11. Summary

| Metric | Score |
|--------|-------|
| CV Accuracy (baseline) | ~0.826 |
| Best CV Accuracy (tuned) | See GridSearchCV output |
| Validation Accuracy | See §8 |
| Validation ROC-AUC | See §8 |

**Top predictive features** (from importance plot):
- `Title_encoded` — captures sex + social status in one feature
- `Sex` — strong survival signal (women prioritized in evacuation)
- `Fare` — proxy for socioeconomic status
- `Age` — children had higher survival rates
- `Pclass` — 1st class passengers had better access to lifeboats